# Socio-Economic Data

# 0 Introduction

## 0 | 1 Summary

This notebook covers the extraction of Socio-Economic statistics from PORDATA website regarding Population, Unemployement, Companies and University(Students) Data by municipality

1. Input <br>
    -Description of Excel files extracted from PORDATA website <br>
2. Output <br>
    Description of the Datasets created by this notebook: <br>
    -2.1. NUTS Dimension table <br>
    -2.2. Population Dataset <br>
    -2.3. Unemployment Dataset <br>
    -2.4. Company Dataset <br>
    -2.5. University Dataset <br>
3. Data Load <br>
        Creation of a function to read PORDATA excel files <br>
        Creating a NUTS dimension table with the 3 levels of NUTS, District and municipality <br>
        Loading the Excel files with the metrics to the notebook <br>
    -3.1. Functions Created  <br>
    -3.2. NUTS Dimension Table <br>
    -3.3. Population Data <br>
    -3.4. Unemployment Data <br>
    -3.5. Companies Data <br>
    -3.6. University Data <br>
4. Creating Datasets <br>
        Creating unified Datasets by domain, bringing together all metrics <br>
    -4.1. Population Dataset <br>
    -4.2. Unemployment Dataset <br>
    -4.3. Company Dataset <br>
    -4.4. University Dataset <br>
5. Dataset Validation <br>
    Validating data <br>
    Creation of a function to check data for each metric by territory and year and another function to visually check <br>
    Checking years with not enough metrics <br>
    Ientifying which metrics to normalize <br>
    -5.1. Functions Created <br>
    -5.2. Validation <br>
    -5.3. Validation Conclusion <br>
6. Dataset Normalization <br>
    Creation of a function to convert metrics to per 10k residents <br>
    Year and P10K Normalization <br>
    -6.1. Functions Created <br>
    -6.2. Year Normalization <br>
    -6.3. P10K Normalization <br>
7. Dataset Export <br>
        -Datasets export to csv files for further analysis <br>

## 0 | 2 Dependencies

In [1]:
# Uncomment and run once to install whats needed
#!pip install openpyxl

In [2]:
import pandas as pd
import numpy as np
import requests
from io import StringIO

# 1 Input

| File                                          | Description         | Source |
|-----------------------------------------------|---------------------|--------|
| 01_PORDATA_Total_Population_with_Residency    | Total number of residents by Municipality | https://www.pordata.pt/municipios/populacao+residente+total-359 |
| 02_PORDATA_Population_Density                 | Density by Municipality (/km2) | https://www.pordata.pt/municipios/populacao+residente+total-359 |
| 03_PORDATA_Purchasing_Power_Per_Capita        | Puschasing Power Per Capita <br> Compared with global averaged to 100 | https://www.pordata.pt/municipios/poder+de+compra+per+capita-118 |
| 04_PORDATA_Unemployment_by_Education_Level    | Unemployment number by Education Level  | https://www.pordata.pt/municipios/desempregados+inscritos+nos+centros+de+emprego+e+de+formacao+profissional+(media+anual)+total+e+por+nivel+de+escolaridade+completo-222|
| 05_PORDATA_Unemployment_by_Gender             | Unemployment number by gender | https://www.pordata.pt/municipios/desempregados+inscritos+nos+centros+de+emprego+e+de+formacao+profissional+(media+anual)+total+e+por+sexo-227 |
| 06_PORDATA_Unemployment_by_Age                | Unemployment number by Age | https://www.pordata.pt/municipios/desempregados+inscritos+nos+centros+de+emprego+e+de+formacao+profissional+(media+anual)+total+e+por+grupo+etario-221 |
| 07_PORDATA_Company_Density                    | Company Density (/km2) <br> Non Financial Companies | https://www.pordata.pt/municipios/densidade+das+empresas+nao+financeiras-920 |
| 08_PORDATA_Company_LegalType                  | Number of companies by legal type <br> Non Financial Companies | https://www.pordata.pt/municipios/empresas+nao+financeiras+total+e+por+forma+juridica-911 |
| 09_PORDATA_Company_ActivitySector             | Number of companies by activity sector <br> Non Financial Companies | https://www.pordata.pt/municipios/empresas+nao+financeiras+total+e+por+setor+de+atividade+economica-346 |
| 10_PORDATA_Company_EmployeeNumber             | Number of companies by number of employees <br> Non Financial Companies | https://www.pordata.pt/municipios/empresas+nao+financeiras+total+e+por+escalao+de+pessoal+ao+servico-915 |
| 11_PORDATA_Company_EmployeesActivitySector    | Number of employees by activity sector <br> Non Financial Companies | https://www.pordata.pt/municipios/pessoal+ao+servico+nas+empresas+nao+financeiras+total+e+por+setor+de+atividade+economica-353 |
| 12_PORDATA_Company_BusinessVolume             | Company Turnover (Business Volume) by activity sector <br> Non Financial Companies | https://www.pordata.pt/municipios/volume+de+negocios+das+empresas+nao+financeiras+total+e+por+setor+de+atividade+economica-589 |
| 13_PORDATA_Company_GVAActivitySector          | Gross Value Added by activity sector <br> Non Financial Companies | https://www.pordata.pt/municipios/valor+acrescentado+bruto+das+empresas+nao+financeiras+total+e+por+setor+de+atividade+economica-588 |
| 14_PORDATA_Company_Births_ActivitySector      | Number of Company births by activity sector <br> Non Financial Companies | https://www.pordata.pt/municipios/nascimentos+de+empresas+nao+financeiras+total+e+por+setor+de+atividade+economica-899 |
| 15_PORDATA_UniversityStudents_Type            | Number of University students by type | https://www.pordata.pt/municipios/alunos+matriculados+no+ensino+superior+total+e+por+tipo+de+ensino-302 |
| 16_PORDATA_UniversityStudentsPublic_Type      | Number of Public University students by type | https://www.pordata.pt/municipios/alunos+matriculados+no+ensino+superior+publico+total+e+por+tipo+de+ensino-303 |
| 17_PORDATA_UniversityStudentsPrivate_Type     | Number of Private University students by type | https://www.pordata.pt/municipios/alunos+matriculados+no+ensino+superior+privado+total+e+por+tipo+de+ensino-304 |
| 18_PORDATA_UniversityStudents_GradLevel       | Number of University students by graduation level | https://www.pordata.pt/municipios/alunos+matriculados+no+ensino+superior+total+e+por+nivel+de+formacao-310 |
| 19_PORDATA_UniversityStudents_Area            | Number of University students by Area | https://www.pordata.pt/municipios/alunos+matriculados+no+ensino+superior+total+e+por+area+de+educacao+e+formacao-313 |


Others
| File                                          | Description         | Source |
|-----------------------------------------------|---------------------|-------------|
|                                               | Employers by Education Level <br> Excluded: no data before 2019 | https://www.pordata.pt/municipios/empregadores+total+e+por+nivel+de+escolaridade-287 |

*****************************************************************************

# 2 Output

## 2 | 1 NUTS Dimension Table

    df_nuts
| table     | column        | description |
|-----------|---------------|-------------|
| df_nuts   | [nuts_1_code] | NUTS level 1 Code |
| df_nuts   | [nuts_1]      | NUTS level 1 Name |
| df_nuts   | [nuts_2_code] | NUTS level 2 Code |
| df_nuts   | [nuts_2]      | NUTS level 2 Name |
| df_nuts   | [nuts_3_code] | NUTS level 3 Code |
| df_nuts   | [nuts_3]      | NUTS level 3 Name |
| df_nuts   | [region]      | District |
| df_nuts   | [territory]   | Municipality |

## 2 | 2 Population Dataset

    df_population
| table                 | column                  | description |
|-----------------------|-------------------------|-------------|
| df_population_clean   | [territory]             | Municipality |
| df_population_clean   | [year]                  | Year of reference |
| df_population_clean   | [population_total]      | Total Number of Resident Population |
| df_population_clean   | [population_density]    | Number of Resident Population per km2 |
| df_population_clean   | [population_pp_pc]      | Purchasing Power per Capita <br> referencing global average as 100 |

## 2 | 3 Unemployment Dataset

       df_unemployment / df_unemployment_p10k

| table                 | column                  | description | notes | p10k |
|-----------------------|-------------------------|-------------|-------|------|
| df_unemployment_clean | [territory] | Municipality |  | Y |
| df_unemployment_clean | [year] | Year of reference |  | Y |
| df_unemployment_clean | [unemployment_total] | Total Number of Unemployed Individuals |  | Y |
| df_unemployment_clean | [unemployment_female] | Number of Unemployed Female Individuals |  | Y |
| df_unemployment_clean | [unemployment_male] | Number of Unemployed Male Individuals |  | Y |
| df_unemployment_clean | [unemployment[<25]] | Number of Unemployed Individuals Aged Under 25 | | Y |
| df_unemployment_clean | [unemployment[25-34]] | Number of Unemployed Individuals Aged 25 to 34 | | Y |
| df_unemployment_clean | [unemployment[35-44]] | Number of Unemployed Individuals Aged 35 to 44 | | Y |
| df_unemployment_clean | [unemployment[45-54]] | Number of Unemployed Individuals Aged 45 to 54 | | Y |
| df_unemployment_clean | [unemployment[>54]] | Number of Unemployed Individuals Aged Over 54 | | Y |
| df_unemployment_clean | [unemployment_el_b1] | Number of Unemployed Individuals with Basic Education - 1st Cycle | Básico / 1º ciclo <br> Basic Education / 1st Cycle | Y |
| df_unemployment_clean | [unemployment_el_b2] | Number of Unemployed Individuals with Basic Education - 2nd Cycle | Básico / 2º ciclo <br> Basic Education / 2nd Cycle | Y |
| df_unemployment_clean | [unemployment_el_b3] | Number of Unemployed Individuals with Basic Education - 3rd Cycle | Básico / 3º ciclo <br> Basic Education / 3rd Cycle | Y |
| df_unemployment_clean | [unemployment_el_hs] | Number of Unemployed Individuals with Secondary Education | Secundário <br> Secondary Education | Y |
| df_unemployment_clean | [unemployment_el_c] | Number of Unemployed Individuals with Higher Education | Superior <br> Higher Education | Y |
| df_unemployment_clean | [unemployment_el_ne] | Number of Unemployed Individuals with No Educational Attainment | Sem nível de escolaridade <br> No Educational Attainment | Y |




## 2 | 4 Company Dataset

       df_company / df_company_p10k

| table | column | description | Notes | p10k |
|-------|--------|-------------|-------|------|
| df_company_clean | [territory] | Municipality |  | Y |
| df_company_clean | [year] | Year of reference |  | Y |
| df_company_clean | [company_total] | Total Number of Companies |  | Y |
| df_company_clean | [company_individual] | Number of Individual Companies |  | Y |
| df_company_clean | [company_society] | Number of Corporate Companies |  | Y |
| df_company_clean | [company_density] | Number of Companies per km2 |  | Y <br> not transformed |
| df_company_clean | [company_nr_cae_a] | Number of Companies in CAE Section A | Agricultura, produção animal, caça, floresta e pesca <br> Agriculture, animal production, hunting, forestry and fishing | Y |
| df_company_clean | [company_nr_cae_b] | Number of Companies in CAE Section B | Indústrias extrativas <br> Mining and quarrying | Y |
| df_company_clean | [company_nr_cae_c] | Number of Companies in CAE Section C | Indústrias transformadoras <br> Manufacturing industries | Y |
| df_company_clean | [company_nr_cae_d] | Number of Companies in CAE Section D | Eletricidade, gás, vapor, água quente e fria e ar frio <br> Electricity, gas, steam, hot and cold water, and air conditioning supply | Y |
| df_company_clean | [company_nr_cae_e] | Number of Companies in CAE Section E | Captação, tratamento e distribuição de água (...) <br> Water collection, treatment and distribution (...) | Y |
| df_company_clean | [company_nr_cae_f] | Number of Companies in CAE Section F | Construção <br> Construction | Y |
| df_company_clean | [company_nr_cae_g] | Number of Companies in CAE Section G | Comércio por grosso e a retalho (...) <br> Wholesale and retail trade (...) | Y |
| df_company_clean | [company_nr_cae_h] | Number of Companies in CAE Section H | Transporte e armazenagem <br> Transportation and storage | Y |
| df_company_clean | [company_nr_cae_i] | Number of Companies in CAE Section I | Alojamento, restauração e similares <br> Accommodation, food services and similar activities | Y |
| df_company_clean | [company_nr_cae_j] | Number of Companies in CAE Section J | Atividade de Informação e comunicação <br> Information and communication activities | Y |
| df_company_clean | [company_nr_cae_l] | Number of Companies in CAE Section L | Atividades imobiliárias <br> Real estate activities | Y |
| df_company_clean | [company_nr_cae_m] | Number of Companies in CAE Section M | Atividades de consultoria, científicas, técnicas e similares <br> Professional, scientific, technical and consultancy activities | Y |
| df_company_clean | [company_nr_cae_n] | Number of Companies in CAE Section N | Atividades administrativas e dos serviços de apoio <br> Administrative and support service activities | Y |
| df_company_clean | [company_nr_cae_p] | Number of Companies in CAE Section P | Educação <br> Education | Y |
| df_company_clean | [company_nr_cae_q] | Number of Companies in CAE Section Q | Atividades de saúde humana e apoio social <br> Human health and social work activities | Y |
| df_company_clean | [company_nr_cae_r] | Number of Companies in CAE Section R | Atividades artísticas, de espetáculos, desportivas e recreativas <br> Arts, entertainment, sports and recreation activities | Y |
| df_company_clean | [company_nr_cae_s] | Number of Companies in CAE Section S | Outras atividades de serviços <br> Other service activities | Y |
| df_company_clean | [company_nr_employee_[<10]] | Number of Companies with Fewer than 10 Employees | Y |
| df_company_clean | [company_nr_employee_[10-19]] | Number of Companies with 10 to 19 Employees | Y |
| df_company_clean | [company_nr_employee_[20-49]] | Number of Companies with 20 to 49 Employees | Y |
| df_company_clean | [company_nr_employee_[50-249]] | Number of Companies with 50 to 249 Employees | Y |
| df_company_clean | [company_nr_employee_[>249]] | Number of Companies with More than 249 Employees | Y |
| df_company_clean | [company_employee_nr_cae_a] | Number of Employees in CAE Section A | Agricultura, produção animal, caça, floresta e pesca <br> Agriculture, animal production, hunting, forestry and fishing | Y |
| df_company_clean | [company_employee_nr_cae_b] | Number of Employees in CAE Section B | Indústrias extrativas <br> Mining and quarrying | Y |
| df_company_clean | [company_employee_nr_cae_c] | Number of Employees in CAE Section C | Indústrias transformadoras <br> Manufacturing industries | Y |
| df_company_clean | [company_employee_nr_cae_d] | Number of Employees in CAE Section D | Eletricidade, gás, vapor, água quente e fria e ar frio <br> Electricity, gas, steam, hot and cold water, and air conditioning supply | Y |
| df_company_clean | [company_employee_nr_cae_e] | Number of Employees in CAE Section E | Captação, tratamento e distribuição de água (...) <br> Water collection, treatment and distribution (...) | Y |
| df_company_clean | [company_employee_nr_cae_f] | Number of Employees in CAE Section F | Construção <br> Construction | Y |
| df_company_clean | [company_employee_nr_cae_g] | Number of Employees in CAE Section G | Comércio por grosso e a retalho (...) <br> Wholesale and retail trade (...) | Y |
| df_company_clean | [company_employee_nr_cae_h] | Number of Employees in CAE Section H | Transporte e armazenagem <br> Transportation and storage | Y |
| df_company_clean | [company_employee_nr_cae_i] | Number of Employees in CAE Section I | Alojamento, restauração e similares <br> Accommodation, food services and similar activities | Y |
| df_company_clean | [company_employee_nr_cae_j] | Number of Employees in CAE Section J | Atividade de Informação e comunicação <br> Information and communication activities | Y |
| df_company_clean | [company_employee_nr_cae_l] | Number of Employees in CAE Section L | Atividades imobiliárias <br> Real estate activities | Y |
| df_company_clean | [company_employee_nr_cae_m] | Number of Employees in CAE Section M | Atividades de consultoria, científicas, técnicas e similares <br> Professional, scientific, technical and consultancy activities | Y |
| df_company_clean | [company_employee_nr_cae_n] | Number of Employees in CAE Section N | Atividades administrativas e dos serviços de apoio <br> Administrative and support service activities | Y |
| df_company_clean | [company_employee_nr_cae_p] | Number of Employees in CAE Section P | Educação <br> Education | Y |
| df_company_clean | [company_employee_nr_cae_q] | Number of Employees in CAE Section Q | Atividades de saúde humana e apoio social <br> Human health and social work activities | Y |
| df_company_clean | [company_employee_nr_cae_r] | Number of Employees in CAE Section R | Atividades artísticas, de espetáculos, desportivas e recreativas <br> Arts, entertainment, sports and recreation activities | Y |
| df_company_clean | [company_employee_nr_cae_s] | Number of Employees in CAE Section S | Outras atividades de serviços <br> Other service activities | Y |
| df_company_clean | [company_employee_nr_total] | Total Number of Employees | Y |
| df_company_clean | [company_gva_cae_a] | Gross Value Added in CAE Section A | Agricultura, produção animal, caça, floresta e pesca <br> Agriculture, animal production, hunting, forestry and fishing | Y |
| df_company_clean | [company_gva_cae_b] | Gross Value Added in CAE Section B | Indústrias extrativas <br> Mining and quarrying | Y |
| df_company_clean | [company_gva_cae_c] | Gross Value Added in CAE Section C | Indústrias transformadoras <br> Manufacturing industries | Y |
| df_company_clean | [company_gva_cae_d] | Gross Value Added in CAE Section D | Eletricidade, gás, vapor, água quente e fria e ar frio <br> Electricity, gas, steam, hot and cold water, and air conditioning supply | Y |
| df_company_clean | [company_gva_cae_e] | Gross Value Added in CAE Section E | Captação, tratamento e distribuição de água (...) <br> Water collection, treatment and distribution (...) | Y |
| df_company_clean | [company_gva_cae_f] | Gross Value Added in CAE Section F | Construção <br> Construction | Y |
| df_company_clean | [company_gva_cae_g] | Gross Value Added in CAE Section G | Comércio por grosso e a retalho (...) <br> Wholesale and retail trade (...) | Y |
| df_company_clean | [company_gva_cae_h] | Gross Value Added in CAE Section H | Transporte e armazenagem <br> Transportation and storage | Y |
| df_company_clean | [company_gva_cae_i] | Gross Value Added in CAE Section I | Alojamento, restauração e similares <br> Accommodation, food services and similar activities | Y |
| df_company_clean | [company_gva_cae_j] | Gross Value Added in CAE Section J | Atividade de Informação e comunicação <br> Information and communication activities | Y |
| df_company_clean | [company_gva_cae_l] | Gross Value Added in CAE Section L | Atividades imobiliárias <br> Real estate activities | Y |
| df_company_clean | [company_gva_cae_m] | Gross Value Added in CAE Section M | Atividades de consultoria, científicas, técnicas e similares <br> Professional, scientific, technical and consultancy activities | Y |
| df_company_clean | [company_gva_cae_n] | Gross Value Added in CAE Section N | Atividades administrativas e dos serviços de apoio <br> Administrative and support service activities | Y |
| df_company_clean | [company_gva_cae_p] | Gross Value Added in CAE Section P | Educação <br> Education | Y |
| df_company_clean | [company_gva_cae_q] | Gross Value Added in CAE Section Q | Atividades de saúde humana e apoio social <br> Human health and social work activities | Y |
| df_company_clean | [company_gva_cae_r] | Gross Value Added in CAE Section R | Atividades artísticas, de espetáculos, desportivas e recreativas <br> Arts, entertainment, sports and recreation activities | Y |
| df_company_clean | [company_gva_cae_s] | Gross Value Added in CAE Section S | Outras atividades de serviços <br> Other service activities | Y |
| df_company_clean | [company_gva_total] | Total Gross Value Added | Y |
| df_company_clean | [company_births_af] | Company Birth Rate - Agriculture and Fishing | Y |
| df_company_clean | [company_births_ice] | Company Birth Rate - Industry, Construction, Energy  | Y |
| df_company_clean | [company_births_s] | Company Birth Rate - Services | Y |
| df_company_clean | [company_births_total] | Total Number of Company Births | Y |

## 2 | 5 University Dataset

    df_university / df_university_p10k
| table | column | description | Notes | p10k |
|-------|--------|-------------|-------|------|
| df_university_clean | [territory] | Municipality |  | Y |
| df_university_clean | [year] | Year of reference |  | Y |
| df_university_clean | [university_total] | Total Number of Higher Education Students |  | Y |
| df_university_clean | [university_poli] | Number of Polytechnic Higher Education Students |  | Y |
| df_university_clean | [university_uni] | Number of University Higher Education Students |  | Y |
| df_university_clean | [university_public_total] | Total Number of Public Higher Education Students |  | Y |
| df_university_clean | [university_public_poli] | Number of Public Polytechnic Higher Education Students |  | Y |
| df_university_clean | [university_public_uni] | Number of Public University Higher Education Students |  | Y |
| df_university_clean | [university_private_total] | Total Number of Private Higher Education Students |  | Y |
| df_university_clean | [university_private_poli] | Number of Private Polytechnic Higher Education Students |  | Y |
| df_university_clean | [university_private_uni] | Number of Private University Higher Education Students |  | Y |
| df_university_clean | [university_gl_lic] | Number of Students Enrolled in Licenciatura Degrees | Licenciatura <br> Bachelor's Degree | Y |
| df_university_clean | [university_gl_lic1] | Number of Students Enrolled in 1st Cycle Degrees | Licenciatura - 1.º ciclo <br> 1st Cycle Bachelor's Degree | Y |
| df_university_clean | [university_gl_ma] | Number of Students Enrolled in Master's Degrees | Mestrado <br> Master's Degree | Y |
| df_university_clean | [university_gl_phd] | Number of Students Enrolled in Doctoral Degrees | Doutoramento <br> Doctorate / PhD | Y |
| df_university_clean | [university_gl_spc] | Number of Students Enrolled in Specialization Programs | Especializações <br> Specializations | Y |
| df_university_clean | [university_gl_fc] | Number of Students Enrolled in Complementary Training Programs | Complemento de Formação <br> Complementary Training | Y |
| df_university_clean | [university_gl_ba] | Number of Students Enrolled in Bacharelato Degrees | Bacharelato <br> Bachelor's Degree (Pre-Bologna) | Y |
| df_university_clean | [university_gl_htc] | Number of Students Enrolled in Higher Technical Professional Courses | Curso técnico superior profissional <br> Higher Technical Course | Y |
| df_university_clean | [university_sa_a] | Number of Students in Agriculture Study Areas | Agricultura <br> Agriculture | Y |
| df_university_clean | [university_sa_ah] | Number of Students in Arts and Humanities Study Areas | Artes e Humanidades <br> Arts and Humanities | Y |
| df_university_clean | [university_sa_sscl] | Number of Students in Social Sciences, Business and Law Study Areas | Ciências Sociais, Comércio e Direito <br> Social Sciences, Business and Law | Y |
| df_university_clean | [university_sa_smi] | Number of Students in Science, Mathematics and Informatics Study Areas | Ciências, Matemática e Informática <br> Science, Mathematics and Informatics | Y |
| df_university_clean | [university_sa_e] | Number of Students in Education Study Areas | Educação <br> Education | Y |
| df_university_clean | [university_sa_eic] | Number of Students in Engineering, Manufacturing and Construction Study Areas | Engenharia, Indústrias Transformadoras e Construção <br> Engineering, Manufacturing and Construction | Y |
| df_university_clean | [university_sa_hsp] | Number of Students in Health and Social Protection Study Areas | Saúde e Proteção Social <br> Health and Social Protection | Y |
| df_university_clean | [university_sa_s] | Number of Students in Services Study Areas | Serviços <br> Services | Y |

# 3 Data Load

## 3 | 1 Data Load | Functions created

In [3]:
# Function to read PORDATA Excel files:
def transform_excel_to_long_df(file_path):

    # read excel file
    df = pd.read_excel(file_path, sheet_name="Quadro")

    # Get the row index of the row that in the first column == "Âmbito Geográfico"
    row_index = df.index[df.iloc[:, 0] == "Âmbito Geográfico"][0]

    # Select that row and the previous one to create column names
    df_colnames = df.loc[[row_index - 1, row_index]].copy()

    # Forward fill only the first row horizontally
    df_colnames.iloc[0] = df_colnames.iloc[0].ffill()

    # Create a new row by concatenating the two rows column-wise
    concat_row = (
        df_colnames.iloc[0].astype(str).fillna('') + "_" +
        df_colnames.iloc[1].astype(str).fillna('')
    )

    df_colnames.loc[len(df_colnames)] = concat_row

    # Set the 3rd row as column names
    df.columns = df_colnames.iloc[2]

    # Remove all rows above row_index including row_index itself
    df = df.iloc[row_index + 1:].reset_index(drop=True)

    # remove not needed bottom rows
    first_null_index = df[df["nan_Âmbito Geográfico"].isna()].index[0]
    df = df.loc[:first_null_index - 1]

    # remove empty columns
    df = df.dropna(axis=1, how='all')

    # rename columns
    df = df.rename(columns={
        "nan_Âmbito Geográfico": "geographic_level",
        "nan_Anos": "territory"
    })

    # Columns that identify each row
    id_vars = ['geographic_level', 'territory']

    # Convert from wide to long
    long_df = df.melt(
        id_vars=id_vars,
        var_name='variable_year',
        value_name='value'
    )

    # Split column names into variable and year
    long_df[['variable', 'Year']] = (
        long_df['variable_year']
        .str.rsplit('_', n=1, expand=True)
    )

    # Pivot variables back to columns
    long_df = (
        long_df
        .pivot_table(
            index=id_vars + ['Year'],
            columns='variable',
            values='value',
            aggfunc='first'
        )
        .reset_index()
    )

    # Remove column index name
    long_df.columns.name = None

    return long_df

## 3 | 2 NUTS Dimension Table

In [4]:
url = "https://pt.wikipedia.org/wiki/Lista_de_munic%C3%ADpios_de_Portugal_por_NUTS,_distritos_e_ilhas"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
print(response.status_code)
tables_mun = pd.read_html(StringIO(response.text))

200


In [5]:
#tabelamestrenuts
df_nuts = pd.concat([tables_mun[1], tables_mun[2], tables_mun[3]])
df_nuts = df_nuts.iloc[:, :5]
df_nuts = df_nuts.rename(columns={
    "Nível I": "nuts_1",
    "Nível II": "nuts_2",
    "Nível III": "nuts_3",
    "Distrito": "region",
    "Municípios": "territory"
})
df_nuts["territory"] = df_nuts["territory"].str.replace(
    r"\[\d+\]",
    "",
    regex=True
)
df_nuts["territory"] = df_nuts["territory"].str.strip()
df_nuts.info()

<class 'pandas.core.frame.DataFrame'>
Index: 308 entries, 0 to 10
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   nuts_1     308 non-null    object
 1   nuts_2     308 non-null    object
 2   nuts_3     308 non-null    object
 3   region     278 non-null    object
 4   territory  308 non-null    object
dtypes: object(5)
memory usage: 14.4+ KB


In [6]:
# Reverse dictionaries (name -> code)

nuts1_dict = {
    "Portugal Continental": "PT1",
    "Região Autónoma dos Açores": "PT2",
    "Região Autónoma da Madeira": "PT3"
}

nuts2_dict = {
    "Região do Norte": "PT11",
    "Algarve": "PT15",
    "Região do Centro": "PT19",
    "Península de Setúbal": "PT1A",
    "Região de Lisboa": "PT1B",
    "Alentejo": "PT1C",
    "Oeste e Vale do Tejo": "PT1D",
    "Região Autónoma dos Açores": "PT20",
    "Região Autónoma da Madeira": "PT30"
}

nuts3_dict = {
    "Alto Minho": "PT111",
    "Cávado": "PT112",
    "Ave": "PT119",
    "Área Metropolitana do Porto": "PT11A",
    "Alto Tâmega": "PT11B",
    "Tâmega e Sousa": "PT11C",
    "Douro": "PT11D",
    "Terras de Trás-os-Montes": "PT11E",
    "Algarve": "PT150",
    "Região de Aveiro": "PT191",
    "Região de Coimbra": "PT192",
    "Região de Leiria": "PT193",
    "Viseu Dão Lafões": "PT194",
    "Beira Baixa": "PT195",
    "Beiras e Serra da Estrela": "PT196",
    "Península de Setúbal": "PT1A0",
    "Área Metropolitana de Lisboa": "PT1B0",
    "Alentejo Litoral": "PT1C1",
    "Baixo Alentejo": "PT1C2",
    "Alto Alentejo": "PT1C3",
    "Alentejo Central": "PT1C4",
    "Oeste": "PT1D1",
    "Médio Tejo": "PT1D2",
    "Lezíria do Tejo": "PT1D3",
    "Região Autónoma dos Açores": "PT200",
    "Região Autónoma da Madeira": "PT300"
}

In [7]:
df_nuts["nuts_1_code"] = df_nuts["nuts_1"].map(nuts1_dict)
df_nuts["nuts_2_code"] = df_nuts["nuts_2"].map(nuts2_dict)
df_nuts["nuts_3_code"] = df_nuts["nuts_3"].map(nuts3_dict)

df_nuts.loc[
    (df_nuts["region"].isna()) & (df_nuts["nuts_3_code"] == "PT300"),
    "region"
] = "Região Autónoma da Madeira"

df_nuts.loc[
    (df_nuts["region"].isna()) & (df_nuts["nuts_3_code"] == "PT200"),
    "region"
] = "Região Autónoma dos Açores"

In [8]:
df_nuts = df_nuts[[
"nuts_1_code",
"nuts_1",
"nuts_2_code",
"nuts_2",
"nuts_3_code",
"nuts_3",
"region",
"territory",
]]


In [9]:
df_nuts

,nuts_1_code,nuts_1,nuts_2_code,nuts_2,nuts_3_code,nuts_3,region,territory
0,PT1,Portugal Continental,PT11,Região do Norte,PT111,Alto Minho,Viana do Castelo,Arcos de Valdevez
1,PT1,Portugal Continental,PT11,Região do Norte,PT111,Alto Minho,Viana do Castelo,Caminha
2,PT1,Portugal Continental,PT11,Região do Norte,PT111,Alto Minho,Viana do Castelo,Melgaço
3,PT1,Portugal Continental,PT11,Região do Norte,PT111,Alto Minho,Viana do Castelo,Monção
4,PT1,Portugal Continental,PT11,Região do Norte,PT111,Alto Minho,Viana do Castelo,Paredes de Coura
...,...,...,...,...,...,...,...,...
6,PT3,Região Autónoma da Madeira,PT30,Região Autónoma da Madeira,PT300,Região Autónoma da Madeira,Região Autónoma da Madeira,Porto Santo
7,PT3,Região Autónoma da Madeira,PT30,Região Autónoma da Madeira,PT300,Região Autónoma da Madeira,Região Autónoma da Madeira,Ribeira Brava
8,PT3,Região Autónoma da Madeira,PT30,Região Autónoma da Madeira,PT300,Região Autónoma da Madeira,Região Autónoma da Madeira,Santa Cruz
9,PT3,Região Autónoma da Madeira,PT30,Região Autónoma da Madeira,PT300,Região Autónoma da Madeira,Região Autónoma da Madeira,Santana


## 3 | 3 Population

### Population | Resident Population

In [10]:
file_path_respop = "../data/raw_data/01_PORDATA_Total_Population_with_Residency.xlsx"
df_respop = transform_excel_to_long_df(file_path_respop)


C:\Users\JPNOVA\AppData\Local\Temp\ipykernel_1760\460686944.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_colnames.iloc[0] = df_colnames.iloc[0].ffill()


In [11]:
#Treating the resulting dataframe
# Rename columns
df_respop = df_respop.rename(columns={
    "nan": "population_total",
    "Year": "year"
})
# Casting columns to correct type
df_respop["year"] = df_respop["year"].astype(int)
df_respop["population_total"] = pd.to_numeric(df_respop["population_total"], errors="coerce")
# Selecting decided type and timeframe
df_respop = df_respop[(df_respop["geographic_level"] == "Município") &(df_respop["year"] >= 2006)]
# Dropping uneeded column
df_respop = df_respop.drop(columns=["geographic_level"])

In [12]:
df_respop

,territory,year,population_total
24,Abrantes,2006,40817
25,Abrantes,2007,40534
26,Abrantes,2008,40255
27,Abrantes,2009,39959
28,Abrantes,2010,39637
...,...,...,...
13239,Óbidos,2020,11875
13240,Óbidos,2021,12224
13241,Óbidos,2022,12619
13242,Óbidos,2023,13061


### Population | Population Density

In [13]:
file_path_popden = "../data/raw_data/02_PORDATA_Population_Density.xlsx"
df_popden = transform_excel_to_long_df(file_path_popden)


C:\Users\JPNOVA\AppData\Local\Temp\ipykernel_1760\460686944.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_colnames.iloc[0] = df_colnames.iloc[0].ffill()


In [14]:
#Treating the resulting dataframe
# Rename columns
df_popden = df_popden.rename(columns={
    "nan": "population_density",
    "Year": "year"
})
# Casting columns to correct type
df_popden["year"] = df_popden["year"].astype(int)
df_popden["population_density"] = pd.to_numeric(df_popden["population_density"], errors="coerce")
# Selecting decided type and timeframe
df_popden = df_popden[(df_popden["geographic_level"] == "Município") &(df_popden["year"] >= 2006)]
# Dropping unneeded column
df_popden = df_popden.drop(columns=["geographic_level"])

In [15]:
df_popden

,territory,year,population_density
0,Abrantes,2009,55.7
1,Abrantes,2010,55.2
2,Abrantes,2011,54.4
3,Abrantes,2012,53.5
4,Abrantes,2013,52.7
...,...,...,...
4923,Óbidos,2020,85.0
4924,Óbidos,2021,87.7
4925,Óbidos,2022,90.6
4926,Óbidos,2023,93.9


### Population | Purchasing Power Per Capita

In [16]:
file_path_ppower_pca = "../data/raw_data/03_PORDATA_Purchasing_Power_Per_Capita.xlsx"
df_ppower_pca = transform_excel_to_long_df(file_path_ppower_pca)


C:\Users\JPNOVA\AppData\Local\Temp\ipykernel_1760\460686944.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_colnames.iloc[0] = df_colnames.iloc[0].ffill()


In [17]:
#Treating the resulting dataframe
# Rename columns
df_ppower_pca = df_ppower_pca.rename(columns={
    "nan": "population_pp_pc",
    "Year": "year"
})
# Casting columns to correct type
df_ppower_pca["year"] = df_ppower_pca["year"].astype(int)
df_ppower_pca["population_pp_pc"] = pd.to_numeric(df_ppower_pca["population_pp_pc"], errors="coerce")
# Selecting decided type and timeframe
df_ppower_pca = df_ppower_pca[(df_ppower_pca["geographic_level"] == "Município") &(df_ppower_pca["year"] >= 2006)]
# Dropping uneeded column
df_ppower_pca = df_ppower_pca.drop(columns=["geographic_level"])

In [18]:
df_ppower_pca

,territory,year,population_pp_pc
3,Abrantes,2007,86.9
4,Abrantes,2009,86.0
5,Abrantes,2011,86.8
6,Abrantes,2013,90.4
7,Abrantes,2015,91.5
...,...,...,...
3691,Óbidos,2015,77.7
3692,Óbidos,2017,75.5
3693,Óbidos,2019,75.9
3694,Óbidos,2021,81.3


## 3 | 4 Unemployment

### Unemployment | Education Level

In [19]:
file_path_unemp_el = "../data/raw_data/04_PORDATA_Unemployment_by_Education_Level.xlsx"
df_unemp_el = transform_excel_to_long_df(file_path_unemp_el)


In [20]:
#Treating the resulting dataframe
# Rename columns
df_unemp_el = df_unemp_el.rename(columns={
    "Básico / 1º ciclo": "unemployment_el_b1",
    "Básico / 2º ciclo": "unemployment_el_b2",
    "Básico / 3º ciclo": "unemployment_el_b3",
    "Secundário": "unemployment_el_hs",
    "Sem nível de escolaridade": "unemployment_el_ne",
    "Superior": "unemployment_el_c",
    "Year": "year",
    "Total": "total"
})


# Casting columns to correct type
df_unemp_el["year"] = df_unemp_el["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_unemp_el.iloc[:, 3:] = df_unemp_el.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_unemp_el = df_unemp_el[(df_unemp_el["geographic_level"] == "Município") &(df_unemp_el["year"] >= 2006)]
# Dropping uneeded column
df_unemp_el = df_unemp_el.drop(columns=["geographic_level", "total"])


In [21]:
df_unemp_el

,territory,year,unemployment_el_b1,unemployment_el_b2,unemployment_el_b3,unemployment_el_hs,unemployment_el_ne,unemployment_el_c
0,Abrantes,2010,539.0,413.2,497.4,408.8,112.6,189.4
1,Abrantes,2011,514.9,394.7,561.5,495.3,96.7,220.9
2,Abrantes,2012,524.8,448.9,677.2,646.0,95.3,293.6
3,Abrantes,2013,538.3,429.5,636.6,629.0,114.0,301.7
4,Abrantes,2014,489.4,386.7,568.8,531.3,120.7,243.4
...,...,...,...,...,...,...,...,...
4923,Óbidos,2021,33.8,32.3,57.7,101.4,3.7,43.3
4924,Óbidos,2022,30.8,28.3,46.3,71.8,3.8,36.6
4925,Óbidos,2023,27.5,25.1,43.0,74.3,6.7,31.9
4926,Óbidos,2024,22.1,24.4,35.0,81.5,6.2,32.8


### Unemployment | Gender

In [22]:
file_path_unemp_g = "../data/raw_data/05_PORDATA_Unemployment_by_Gender.xlsx"
df_unemp_g = transform_excel_to_long_df(file_path_unemp_g)


In [23]:
#Treating the resulting dataframe
# Rename columns
df_unemp_g = df_unemp_g.rename(columns={
    "Feminino": "unemployment_female",
    "Masculino": "unemployment_male",
    "Year": "year",
    "Total": "unemployment_total"
})

# Casting columns to correct type
df_unemp_g["year"] = df_unemp_g["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_unemp_g.iloc[:, 3:] = df_unemp_g.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_unemp_g = df_unemp_g[(df_unemp_g["geographic_level"] == "Município") &(df_unemp_g["year"] >= 2006)]
# Dropping uneeded column
df_unemp_g = df_unemp_g.drop(columns=["geographic_level"])


In [24]:
df_unemp_g

,territory,year,unemployment_female,unemployment_male,unemployment_total
2,Abrantes,2009,1178.6,985.5,2164.1
3,Abrantes,2010,1107.9,1052.5,2160.4
4,Abrantes,2011,1170.0,1114.0,2284.0
5,Abrantes,2012,1380.4,1305.3,2685.8
6,Abrantes,2013,1324.8,1324.3,2649.0
...,...,...,...,...,...
5847,Óbidos,2021,168.3,103.9,272.2
5848,Óbidos,2022,134.2,83.5,217.7
5849,Óbidos,2023,126.1,82.3,208.4
5850,Óbidos,2024,117.7,84.3,202.0


### Unemployment | Age

In [25]:
file_path_unemp_a = "../data/raw_data/06_PORDATA_Unemployment_by_Age.xlsx"
df_unemp_a = transform_excel_to_long_df(file_path_unemp_a)


In [26]:
#Treating the resulting dataframe
# Rename columns
df_unemp_a = df_unemp_a.rename(columns={
    "25-34": "unemployment[25-34]",
    "35-44": "unemployment[35-44]",
    "45-54": "unemployment[45-54]",
    "55 ou mais": "unemployment[>54]",
    "Menos de 25": "unemployment[<25]",
    "Year": "year",
    "Total": "total"
})


# Casting columns to correct type
df_unemp_a["year"] = df_unemp_a["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_unemp_a.iloc[:, 3:] = df_unemp_a.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_unemp_a = df_unemp_a[(df_unemp_a["geographic_level"] == "Município") &(df_unemp_a["year"] >= 2006)]
# Dropping uneeded column
df_unemp_a = df_unemp_a.drop(columns=["geographic_level", "total"])


In [27]:
df_unemp_a

,territory,year,unemployment[25-34],unemployment[35-44],unemployment[45-54],unemployment[>54],unemployment[<25]
2,Abrantes,2009,589.7,439.2,424.8,379.3,331.1
3,Abrantes,2010,537.0,469.0,469.8,377.7,306.9
4,Abrantes,2011,581.1,519.3,471.6,411.8,300.3
5,Abrantes,2012,696.6,661.1,521.5,433.3,373.3
6,Abrantes,2013,629.2,628.1,558.9,485.7,347.2
...,...,...,...,...,...,...,...
5847,Óbidos,2021,44.2,52.3,67.1,82.1,26.5
5848,Óbidos,2022,29.6,37.7,50.8,75.0,24.6
5849,Óbidos,2023,26.6,42.8,43.8,72.2,23.1
5850,Óbidos,2024,31.5,39.1,41.6,68.3,21.6


## 3 | 5 Companies

### Company | Density

In [28]:
file_path_company_den = "../data/raw_data/07_PORDATA_Company_Density.xlsx"
df_company_den = transform_excel_to_long_df(file_path_company_den)


C:\Users\JPNOVA\AppData\Local\Temp\ipykernel_1760\460686944.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_colnames.iloc[0] = df_colnames.iloc[0].ffill()


In [29]:
#Treating the resulting dataframe
# Rename columns
df_company_den = df_company_den.rename(columns={
    "nan": "company_density",
    "Year": "year"
})

# Casting columns to correct type
df_company_den["year"] = df_company_den["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_company_den.iloc[:, 3:] = df_company_den.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_company_den = df_company_den[(df_company_den["geographic_level"] == "Município") &(df_company_den["year"] >= 2006)]
# Dropping uneeded column
df_company_den = df_company_den.drop(columns=["geographic_level"])


In [30]:
df_company_den

,territory,year,company_density
0,Abrantes,2009,4.6
1,Abrantes,2010,4.3
2,Abrantes,2011,4.3
3,Abrantes,2012,4.1
4,Abrantes,2013,4.0
...,...,...,...
4923,Óbidos,2020,13.2
4924,Óbidos,2021,14.0
4925,Óbidos,2022,15.9
4926,Óbidos,2023,16.6


### Company | Legal Type

In [31]:
file_path_company_lt = "../data/raw_data/08_PORDATA_Company_LegalType.xlsx"
df_company_lt = transform_excel_to_long_df(file_path_company_lt)


In [32]:
#Treating the resulting dataframe
# Rename columns
df_company_lt = df_company_lt.rename(columns={
    "Individual": "company_individual",
    "Sociedades": "company_society",
    "Year": "year",
    "Total": "company_total"
})

# Casting columns to correct type
df_company_lt["year"] = df_company_lt["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_company_lt.iloc[:, 3:] = df_company_lt.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_company_lt = df_company_lt[(df_company_lt["geographic_level"] == "Município") &(df_company_lt["year"] >= 2006)]
# Dropping uneeded column
df_company_lt = df_company_lt.drop(columns=["geographic_level"])


In [33]:
df_company_lt

,territory,year,company_individual,company_society,company_total
0,Abrantes,2009,2327,971,3298
1,Abrantes,2010,2147,932,3079
2,Abrantes,2011,2143,938,3081
3,Abrantes,2012,2002,920,2922
4,Abrantes,2013,1938,900,2838
...,...,...,...,...,...
4923,Óbidos,2020,1281,581,1862
4924,Óbidos,2021,1386,598,1984
4925,Óbidos,2022,1623,634,2257
4926,Óbidos,2023,1682,661,2343


### Company | Activity Sector

In [34]:
file_path_company_as1 = "../data/raw_data/09_PORDATA_Company_ActivitySector_part1.xlsx"
file_path_company_as2 = "../data/raw_data/09_PORDATA_Company_ActivitySector_part2.xlsx"
df_company_as1 = transform_excel_to_long_df(file_path_company_as1)
df_company_as2 = transform_excel_to_long_df(file_path_company_as2)
# Union of the 2 dataframes
df_company_as = pd.concat([df_company_as1, df_company_as2], ignore_index=True)

In [35]:
#Treating the resulting dataframe
# Rename columns
df_company_as = df_company_as.rename(columns={
    "Agricultura, produção animal, caça, floresta e pesca": "company_nr_cae_a",
    "Indústrias extrativas": "company_nr_cae_b",
    "Indústrias transformadoras": "company_nr_cae_c",
    "Eletricidade, gás, vapor, água quente e fria e ar frio": "company_nr_cae_d",
    "Captação, tratamento e distribuição de água (...)": "company_nr_cae_e",
    "Construção": "company_nr_cae_f",
    "Comércio por grosso e a retalho (...)": "company_nr_cae_g",
    "Transporte e armazenagem": "company_nr_cae_h",
    "Alojamento, restauração e similares": "company_nr_cae_i",
    "Atividade de Informação e comunicação": "company_nr_cae_j",
    "Atividades imobiliárias": "company_nr_cae_l",
    "Atividades de consultoria, científicas, técnicas e similares": "company_nr_cae_m",
    "Atividades administrativas e dos serviços de apoio": "company_nr_cae_n",
    "Educação": "company_nr_cae_p",
    "Atividades de saúde humana e apoio social": "company_nr_cae_q",
    "Atividades artísticas, de espetáculos, desportivas e recreativas": "company_nr_cae_r",
    "Outras atividades de serviços": "company_nr_cae_s",
    "Year": "year",
    "Total": "total"
})


# Casting columns to correct type
df_company_as["year"] = df_company_as["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_company_as.iloc[:, 3:] = df_company_as.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_company_as = df_company_as[(df_company_as["geographic_level"] == "Município") &(df_company_as["year"] >= 2006)]
# Dropping uneeded column
df_company_as = df_company_as.drop(columns=["geographic_level", "total"])


In [36]:
df_company_as

,territory,year,company_nr_cae_a,company_nr_cae_i,company_nr_cae_j,company_nr_cae_n,company_nr_cae_r,company_nr_cae_m,company_nr_cae_q,company_nr_cae_l,company_nr_cae_e,company_nr_cae_g,company_nr_cae_f,company_nr_cae_p,company_nr_cae_d,company_nr_cae_b,company_nr_cae_c,company_nr_cae_s,company_nr_cae_h
0,Abrantes,2019,304,269,36,292,92,242,304,53,8,653,207,161,51,0,159,188,65
1,Abrantes,2020,299,263,41,313,79,258,314,63,9,626,205,168,55,0,147,185,66
2,Abrantes,2021,269,263,44,291,83,280,327,70,8,612,211,138,46,0,156,193,72
3,Abrantes,2022,290,276,45,341,93,289,346,84,8,611,210,160,54,0,160,189,73
4,Abrantes,2023,286,271,57,378,106,289,342,88,8,603,225,169,37,0,172,212,74
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3979,Óbidos,2014,342,179,29,179,39,106,52,31,4,293,121,67,2,1,84,51,20
3980,Óbidos,2015,347,198,34,191,42,115,57,45,3,291,125,62,3,1,79,54,18
3981,Óbidos,2016,360,253,32,187,44,132,58,52,3,303,136,63,13,0,93,54,19
3982,Óbidos,2017,367,261,38,198,47,137,64,52,3,304,143,58,14,0,96,54,20


### Company | Number of Employees

In [37]:
file_path_company_empnr = "../data/raw_data/10_PORDATA_Company_EmployeeNumber.xlsx"
df_company_empnr = transform_excel_to_long_df(file_path_company_empnr)

In [38]:
#Treating the resulting dataframe
# Rename columns
df_company_empnr = df_company_empnr.rename(columns={
    "10-19": "company_nr_employee_[10-19]",
    "20-49": "company_nr_employee_[20-49]",
    "250 ou mais": "company_nr_employee_[>249]",
    "50-249": "company_nr_employee_[50-249]",
    "Menos de 10": "company_nr_employee_[<10]",
    "Year": "year",
    "Total": "total",
})
# Casting columns to correct type
df_company_empnr["year"] = df_company_empnr["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_company_empnr.iloc[:, 3:] = df_company_empnr.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_company_empnr = df_company_empnr[(df_company_empnr["geographic_level"] == "Município") &(df_company_empnr["year"] >= 2006)]
# Dropping uneeded column
df_company_empnr = df_company_empnr.drop(columns=["geographic_level","total"])


In [39]:
df_company_empnr

,territory,year,company_nr_employee_[10-19],company_nr_employee_[20-49],company_nr_employee_[>249],company_nr_employee_[50-249],company_nr_employee_[<10]
0,Abrantes,2009,64,39,2,15,3178
1,Abrantes,2010,66,36,2,13,2962
2,Abrantes,2011,51,37,2,12,2979
3,Abrantes,2012,47,35,2,10,2828
4,Abrantes,2013,52,32,1,10,2743
...,...,...,...,...,...,...,...
4923,Óbidos,2020,34,24,0,6,1798
4924,Óbidos,2021,32,25,0,6,1921
4925,Óbidos,2022,36,22,0,9,2190
4926,Óbidos,2023,36,24,0,11,2272


### Company | Employee Number by Activity Sector

In [40]:
file_path_company_emp_as1 = "../data/raw_data/11_PORDATA_Company_EmployeesActivitySector_part1.xlsx"
file_path_company_emp_as2 = "../data/raw_data/11_PORDATA_Company_EmployeesActivitySector_part2.xlsx"
df_company_emp_as1 = transform_excel_to_long_df(file_path_company_emp_as1)
df_company_emp_as2 = transform_excel_to_long_df(file_path_company_emp_as2)
# Union of the 2 dataframes
df_company_emp_as = pd.concat([df_company_emp_as1, df_company_emp_as2], ignore_index=True)

In [41]:
#Treating the resulting dataframe
# Rename columns
df_company_emp_as = df_company_emp_as.rename(columns={
    "Agricultura, produção animal, caça, floresta e pesca": "company_employee_nr_cae_a",
    "Indústrias extrativas": "company_employee_nr_cae_b",
    "Indústrias transformadoras": "company_employee_nr_cae_c",
    "Eletricidade, gás, vapor, água quente e fria e ar frio": "company_employee_nr_cae_d",
    "Captação, tratamento e distribuição de água (...)": "company_employee_nr_cae_e",
    "Construção": "company_employee_nr_cae_f",
    "Comércio por grosso e a retalho (...)": "company_employee_nr_cae_g",
    "Transporte e armazenagem": "company_employee_nr_cae_h",
    "Alojamento, restauração e similares": "company_employee_nr_cae_i",
    "Atividade de Informação e comunicação": "company_employee_nr_cae_j",
    "Atividades imobiliárias": "company_employee_nr_cae_l",
    "Atividades de consultoria, científicas, técnicas e similares": "company_employee_nr_cae_m",
    "Atividades administrativas e dos serviços de apoio": "company_employee_nr_cae_n",
    "Educação": "company_employee_nr_cae_p",
    "Atividades de saúde humana e apoio social": "company_employee_nr_cae_q",
    "Atividades artísticas, de espetáculos, desportivas e recreativas": "company_employee_nr_cae_r",
    "Outras atividades de serviços": "company_employee_nr_cae_s",
    "Year": "year",
    "Total": "company_employee_nr_total",
})


# Casting columns to correct type
df_company_emp_as["year"] = df_company_emp_as["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_company_emp_as.iloc[:, 3:] = df_company_emp_as.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_company_emp_as = df_company_emp_as[(df_company_emp_as["geographic_level"] == "Município") &(df_company_emp_as["year"] >= 2006)]
# Dropping uneeded column
df_company_emp_as = df_company_emp_as.drop(columns=["geographic_level"])


In [42]:
df_company_emp_as

,territory,year,company_employee_nr_cae_a,company_employee_nr_cae_i,company_employee_nr_cae_j,company_employee_nr_cae_n,company_employee_nr_cae_r,company_employee_nr_cae_m,company_employee_nr_cae_q,company_employee_nr_cae_l,company_employee_nr_cae_e,company_employee_nr_cae_g,company_employee_nr_cae_f,company_employee_nr_cae_p,company_employee_nr_cae_d,company_employee_nr_cae_b,company_employee_nr_cae_c,company_employee_nr_cae_s,company_employee_nr_total,company_employee_nr_cae_h
0,Abrantes,2019,538,696,66,360,103,421,497,73,181,1600,574,183,187,0,2013,262,7960,206
1,Abrantes,2020,574,620,71,373,90,449,552,86,181,1547,643,186,184,0,1794,265,7807,192
2,Abrantes,2021,462,590,83,350,102,489,566,90,184,1520,668,155,176,0,1893,276,7799,195
3,Abrantes,2022,479,726,0,387,114,550,598,111,0,1561,726,179,139,0,1993,281,8324,207
4,Abrantes,2023,487,663,74,443,126,584,628,122,197,1596,700,189,113,0,2076,307,8514,209
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3979,Óbidos,2014,505,458,113,247,44,259,95,36,129,790,335,109,0,0,316,65,3577,71
3980,Óbidos,2015,512,501,139,231,46,311,77,62,10,838,372,0,3,0,334,67,3664,43
3981,Óbidos,2016,522,695,146,227,83,350,89,75,13,930,428,112,13,0,399,65,4190,43
3982,Óbidos,2017,529,785,180,247,116,352,95,79,12,950,446,111,14,0,439,61,4464,48


### Company | Business Volume by Activity Sector

In [43]:
file_path_company_bvol1 = "../data/raw_data/12_PORDATA_Company_BusinessVolume_part1.xlsx"
file_path_company_bvol2 = "../data/raw_data/12_PORDATA_Company_BusinessVolume_part2.xlsx"
df_company_bvol1 = transform_excel_to_long_df(file_path_company_bvol1)
df_company_bvol2 = transform_excel_to_long_df(file_path_company_bvol2)
# Union of the 2 dataframes
df_company_bvol = pd.concat([df_company_bvol1, df_company_bvol2], ignore_index=True)

In [44]:
#Treating the resulting dataframe
# Rename columns
df_company_bvol = df_company_bvol.rename(columns={
    "Agricultura, produção animal, caça, floresta e pesca": "company_bvol_CAE_A",
    "Indústrias extrativas": "company_bvol_CAE_B",
    "Indústrias transformadoras": "company_bvol_CAE_C",
    "Eletricidade, gás, vapor, água quente e fria e ar frio": "company_bvol_CAE_D",
    "Captação, tratamento e distribuição de água (...)": "company_bvol_CAE_E",
    "Construção": "company_bvol_CAE_F",
    "Comércio por grosso e a retalho (...)": "company_bvol_CAE_G",
    "Transporte e armazenagem": "company_bvol_CAE_H",
    "Alojamento, restauração e similares": "company_bvol_CAE_I",
    "Atividade de Informação e comunicação": "company_bvol_CAE_J",
    "Atividades imobiliárias": "company_bvol_CAE_L",
    "Atividades de consultoria, científicas, técnicas e similares": "company_bvol_CAE_M",
    "Atividades administrativas e dos serviços de apoio": "company_bvol_CAE_N",
    "Educação": "company_bvol_CAE_P",
    "Atividades de saúde humana e apoio social": "company_bvol_CAE_Q",
    "Atividades artísticas, de espetáculos, desportivas e recreativas": "company_bvol_CAE_R",
    "Outras atividades de serviços": "company_bvol_CAE_S",
    "Total": "total",
})


# Casting columns to correct type
df_company_bvol["Year"] = df_company_bvol["Year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_company_bvol.iloc[:, 3:] = df_company_bvol.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_company_bvol = df_company_bvol[(df_company_bvol["geographic_level"] == "Município") &(df_company_bvol["Year"] >= 2006)]
# Dropping uneeded column
df_company_bvol = df_company_bvol.drop(columns=["geographic_level"])


In [45]:
df_company_bvol

,territory,Year,company_bvol_CAE_A,company_bvol_CAE_I,company_bvol_CAE_J,company_bvol_CAE_N,company_bvol_CAE_R,company_bvol_CAE_M,company_bvol_CAE_Q,company_bvol_CAE_L,company_bvol_CAE_E,company_bvol_CAE_G,company_bvol_CAE_F,company_bvol_CAE_P,company_bvol_CAE_D,company_bvol_CAE_B,company_bvol_CAE_C,company_bvol_CAE_S,total,company_bvol_CAE_H
0,Abrantes,2019,23905,22134,1710,3870,1902,12753,13021,2386,30108,144424,24947,1590,193187,0,430264,2762,917479,8517
1,Abrantes,2020,24948,16737,1772,2847,1156,14254,13433,3452,28470,135256,24265,1361,163731,0,327649,3145,770095,7619
2,Abrantes,2021,26984,18681,2023,3122,1618,19982,17499,4396,38566,148420,29952,1485,164896,0,423231,3687,914008,9465
3,Abrantes,2022,30970,29026,0,3489,2511,24761,18247,6546,0,157534,32534,1682,38556,0,510651,4047,912401,11754
4,Abrantes,2023,31136,26550,2479,7241,3083,29383,20609,9099,37128,176111,36625,1868,36238,0,565531,4723,1011444,23641
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3979,Óbidos,2014,19546,14672,15190,3513,1123,15212,1209,1433,33546,105671,11937,820,0,0,28885,958,256626,2509
3980,Óbidos,2015,22637,17813,17556,2810,1285,18306,1266,2355,425,108380,13107,0,8,0,27493,1146,237649,1931
3981,Óbidos,2016,20777,24705,19769,3101,2472,23149,1349,5038,460,113467,14976,976,17,0,30831,1031,263895,1777
3982,Óbidos,2017,26910,30304,11894,3585,3174,21245,1650,5878,407,124568,18883,979,20,0,40041,957,292310,1816


### Company | Gross Value Added (GVA) by Activity Sector

In [46]:
file_path_company_gva_as1 = "../data/raw_data/13_PORDATA_Company_GVAActivitySector_part1.xlsx"
file_path_company_gva_as2 = "../data/raw_data/13_PORDATA_Company_GVAActivitySector_part2.xlsx"
df_company_gva_as1 = transform_excel_to_long_df(file_path_company_gva_as1)
df_company_gva_as2 = transform_excel_to_long_df(file_path_company_gva_as2)
# Union of the 2 dataframes
df_company_gva_as = pd.concat([df_company_gva_as1, df_company_gva_as2], ignore_index=True)

In [47]:
#Treating the resulting dataframe
# Rename columns
df_company_gva_as = df_company_gva_as.rename(columns={
    "Agricultura, produção animal, caça, floresta e pesca": "company_gva_cae_a",
    "Indústrias extrativas": "company_gva_cae_b",
    "Indústrias transformadoras": "company_gva_cae_c",
    "Eletricidade, gás, vapor, água quente e fria e ar frio": "company_gva_cae_d",
    "Captação, tratamento e distribuição de água (...)": "company_gva_cae_e",
    "Construção": "company_gva_cae_f",
    "Comércio por grosso e a retalho (...)": "company_gva_cae_g",
    "Transporte e armazenagem": "company_gva_cae_h",
    "Alojamento, restauração e similares": "company_gva_cae_i",
    "Atividade de Informação e comunicação": "company_gva_cae_j",
    "Atividades imobiliárias": "company_gva_cae_l",
    "Atividades de consultoria, científicas, técnicas e similares": "company_gva_cae_m",
    "Atividades administrativas e dos serviços de apoio": "company_gva_cae_n",
    "Educação": "company_gva_cae_p",
    "Atividades de saúde humana e apoio social": "company_gva_cae_q",
    "Atividades artísticas, de espetáculos, desportivas e recreativas": "company_gva_cae_r",
    "Outras atividades de serviços": "company_gva_cae_s",
    "Year": "year",
    "Total": "company_gva_total",
})


# Casting columns to correct type
df_company_gva_as["year"] = df_company_gva_as["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_company_gva_as.iloc[:, 3:] = df_company_gva_as.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_company_gva_as = df_company_gva_as[(df_company_gva_as["geographic_level"] == "Município") &(df_company_gva_as["year"] >= 2006)]
# Dropping uneeded column
df_company_gva_as = df_company_gva_as.drop(columns=["geographic_level"])


In [48]:
df_company_gva_as

,territory,year,company_gva_cae_a,company_gva_cae_i,company_gva_cae_j,company_gva_cae_n,company_gva_cae_r,company_gva_cae_m,company_gva_cae_q,company_gva_cae_l,company_gva_cae_e,company_gva_cae_g,company_gva_cae_f,company_gva_cae_p,company_gva_cae_d,company_gva_cae_b,company_gva_cae_c,company_gva_cae_s,company_gva_total,company_gva_cae_h
0,Abrantes,2019,6045,8142,1082,2682,506,8254,6937,1507,6779,24496,10588,1069,113379,0,67599,1578,263550,2908
1,Abrantes,2020,7240,5627,1312,2236,315,9932,6949,1915,6031,23903,9916,911,104386,0,55655,1874,241208,3005
2,Abrantes,2021,7578,6365,1450,2239,453,13076,8859,1995,8917,26370,11269,975,94752,0,66070,2076,255817,3370
3,Abrantes,2022,6800,11805,0,2425,653,16744,8985,1857,0,28848,11284,1162,23898,0,77412,2217,206657,4480
4,Abrantes,2023,8275,9782,1630,3259,923,19529,10233,3354,10015,31907,13916,1362,20018,0,81542,2738,224367,5882
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3979,Óbidos,2014,6179,6119,10279,2222,483,7456,805,221,18281,14829,5882,438,0,0,6103,550,80815,878
3980,Óbidos,2015,6764,7296,5162,1672,577,8991,879,125,182,16200,5575,0,6,0,7052,654,62329,753
3981,Óbidos,2016,6545,13202,6424,1845,1173,8872,874,779,199,18101,6953,423,13,0,9208,666,76030,751
3982,Óbidos,2017,8193,13494,5210,2068,2433,6543,1091,2050,220,19829,5974,480,15,0,11340,552,80251,759


### Company | Company birth by Activity Sector

In [49]:
file_path_company_births_as = "../data/raw_data/14_PORDATA_Company_Births_ActivitySector.xlsx"
df_company_births_as = transform_excel_to_long_df(file_path_company_births_as)

In [50]:
#Treating the resulting dataframe
# Rename columns
df_company_births_as = df_company_births_as.rename(columns={
    "Agricultura e Pescas": "company_births_af",
    "Indústrias, Construção e Energia": "company_births_ice",
    "Serviços": "company_births_s",
    "Year": "year",
    "Total": "company_births_total",
})
# Casting columns to correct type
df_company_births_as["year"] = df_company_births_as["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_company_births_as.iloc[:, 3:] = df_company_births_as.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_company_births_as = df_company_births_as[(df_company_births_as["geographic_level"] == "Município") &(df_company_births_as["year"] >= 2006)]
# Dropping uneeded column
df_company_births_as = df_company_births_as.drop(columns=["geographic_level"])


In [51]:
df_company_births_as

,territory,year,company_births_af,company_births_ice,company_births_s,company_births_total
0,Abrantes,2009,14,29,369,412
1,Abrantes,2010,27,25,282,334
2,Abrantes,2011,25,32,299,356
3,Abrantes,2012,23,33,287,343
4,Abrantes,2013,71,31,276,378
...,...,...,...,...,...,...
4923,Óbidos,2020,20,21,134,175
4924,Óbidos,2021,19,22,150,191
4925,Óbidos,2022,11,31,304,346
4926,Óbidos,2023,16,28,321,365


## 3 | 6 University

### University | Students by Type

In [52]:
file_path_unistudents_type = "../data/raw_data/15_PORDATA_UniversityStudents_Type.xlsx"
df_unistudents_type = transform_excel_to_long_df(file_path_unistudents_type)

In [53]:
#Treating the resulting dataframe
# Rename columns
df_unistudents_type = df_unistudents_type.rename(columns={
    "Politécnico": "university_poli",
    "Universitário": "university_uni",
    "Year": "year",
    "Total": "university_total",
})
# Casting columns to correct type
df_unistudents_type["year"] = df_unistudents_type["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_unistudents_type.iloc[:, 3:] = df_unistudents_type.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_unistudents_type = df_unistudents_type[(df_unistudents_type["geographic_level"] == "Município") &(df_unistudents_type["year"] >= 2006)]
# Dropping uneeded column
df_unistudents_type = df_unistudents_type.drop(columns=["geographic_level"])


In [54]:
df_unistudents_type

,territory,year,university_poli,university_total,university_uni
2,Abrantes,2009,455,455,0
3,Abrantes,2010,458,458,0
4,Abrantes,2011,429,429,0
5,Abrantes,2012,402,402,0
6,Abrantes,2013,372,372,0
...,...,...,...,...,...
5847,Óbidos,2021,0,0,0
5848,Óbidos,2022,0,0,0
5849,Óbidos,2023,0,0,0
5850,Óbidos,2024,0,0,0


### University | Students by Type - Public

In [55]:
file_path_unipublicstudents_type = "../data/raw_data/16_PORDATA_UniversityStudentsPublic_Type.xlsx"
df_unipublicstudents_type = transform_excel_to_long_df(file_path_unipublicstudents_type)

In [56]:
#Treating the resulting dataframe
# Rename columns
df_unipublicstudents_type = df_unipublicstudents_type.rename(columns={
    "Politécnico": "university_public_poli",
    "Universitário": "university_public_uni",
    "Year": "year",
    "Total": "university_public_total",
})
# Casting columns to correct type
df_unipublicstudents_type["year"] = df_unipublicstudents_type["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_unipublicstudents_type.iloc[:, 3:] = df_unipublicstudents_type.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_unipublicstudents_type = df_unipublicstudents_type[(df_unipublicstudents_type["geographic_level"] == "Município") &(df_unipublicstudents_type["year"] >= 2006)]
# Dropping uneeded column
df_unipublicstudents_type = df_unipublicstudents_type.drop(columns=["geographic_level"])


In [57]:
df_unipublicstudents_type

,territory,year,university_public_poli,university_public_total,university_public_uni
2,Abrantes,2009,455,455,0
3,Abrantes,2010,458,458,0
4,Abrantes,2011,429,429,0
5,Abrantes,2012,402,402,0
6,Abrantes,2013,372,372,0
...,...,...,...,...,...
5847,Óbidos,2021,0,0,0
5848,Óbidos,2022,0,0,0
5849,Óbidos,2023,0,0,0
5850,Óbidos,2024,0,0,0


### University | Students by Type - Private

In [58]:
file_path_uniprivstudents_type = "../data/raw_data/17_PORDATA_UniversityStudentsPrivate_Type.xlsx"
df_uniprivstudents_type = transform_excel_to_long_df(file_path_uniprivstudents_type)

In [59]:
#Treating the resulting dataframe
# Rename columns
df_uniprivstudents_type = df_uniprivstudents_type.rename(columns={
    "Politécnico": "university_private_poli",
    "Universitário": "university_private_uni",
    "Year": "year",
    "Total": "university_private_total",
})
# Casting columns to correct type
df_uniprivstudents_type["year"] = df_uniprivstudents_type["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_uniprivstudents_type.iloc[:, 3:] = df_uniprivstudents_type.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_uniprivstudents_type = df_uniprivstudents_type[(df_uniprivstudents_type["geographic_level"] == "Município") &(df_uniprivstudents_type["year"] >= 2006)]
# Dropping uneeded column
df_uniprivstudents_type = df_uniprivstudents_type.drop(columns=["geographic_level"])


In [60]:
df_uniprivstudents_type

,territory,year,university_private_poli,university_private_total,university_private_uni
2,Abrantes,2009,0,0,0
3,Abrantes,2010,0,0,0
4,Abrantes,2011,0,0,0
5,Abrantes,2012,0,0,0
6,Abrantes,2013,0,0,0
...,...,...,...,...,...
5847,Óbidos,2021,0,0,0
5848,Óbidos,2022,0,0,0
5849,Óbidos,2023,0,0,0
5850,Óbidos,2024,0,0,0


### University | Students by Grad Level

In [61]:
file_path_unistudents_gradlevel = "../data/raw_data/18_PORDATA_UniversityStudents_GradLevel.xlsx"
df_unistudents_gradlevel = transform_excel_to_long_df(file_path_unistudents_gradlevel)

In [62]:
#Treating the resulting dataframe
# Rename columns
df_unistudents_gradlevel = df_unistudents_gradlevel.rename(columns={
    "Bacharelato": "university_gl_ba",
    "CESE": "university_gl_esp",
    "Complemento de Formação": "university_gl_fc",
    "Curso técnico superior profissional": "university_gl_htc",
    "Doutoramento": "university_gl_phd",
    "Especializações": "university_gl_spc",
    "Licenciatura": "university_gl_lic",
    "Licenciatura - 1.º ciclo": "university_gl_lic1",
    "Mestrado": "university_gl_ma",
    "Mestrado Integrado": "university_gl_im",
    "Year": "year",
    "Total": "total"
})
# Casting columns to correct type
df_unistudents_gradlevel["year"] = df_unistudents_gradlevel["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_unistudents_gradlevel.iloc[:, 3:] = df_unistudents_gradlevel.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_unistudents_gradlevel = df_unistudents_gradlevel[(df_unistudents_gradlevel["geographic_level"] == "Município") &(df_unistudents_gradlevel["year"] >= 2006)]
# Dropping uneeded column
df_unistudents_gradlevel = df_unistudents_gradlevel.drop(columns=["geographic_level","total"])


In [63]:
df_unistudents_gradlevel

,territory,year,university_gl_ba,university_gl_esp,university_gl_fc,university_gl_htc,university_gl_phd,university_gl_spc,university_gl_lic,university_gl_lic1,university_gl_ma,university_gl_im
0,Abrantes,2016,0,0,0,34,0,0,0,260,19,0
1,Abrantes,2017,0,0,0,67,0,0,0,245,1,0
2,Abrantes,2018,0,0,0,76,0,0,0,237,10,0
3,Abrantes,2019,0,0,0,111,0,0,0,234,14,0
4,Abrantes,2020,0,0,0,164,0,0,0,265,6,0
...,...,...,...,...,...,...,...,...,...,...,...,...
3075,Óbidos,2021,0,0,0,0,0,0,0,0,0,0
3076,Óbidos,2022,0,0,0,0,0,0,0,0,0,0
3077,Óbidos,2023,0,0,0,0,0,0,0,0,0,0
3078,Óbidos,2024,0,0,0,0,0,0,0,0,0,0


### University | Students by Study Area

In [64]:
file_path_unistudents_area = "../data/raw_data/19_PORDATA_UniversityStudents_Area.xlsx"
df_unistudents_area = transform_excel_to_long_df(file_path_unistudents_area)

In [65]:
#Treating the resulting dataframe
# Rename columns
df_unistudents_area = df_unistudents_area.rename(columns={
    "Agricultura": "university_sa_a",
    "Artes e Humanidades": "university_sa_ah",
    "Ciências Sociais, Comércio e Direito": "university_sa_sscl",
    "Ciências, Matemática e Informática": "university_sa_smi",
    "Educação": "university_sa_e",
    "Engenharia, Indústrias Transformadoras e Construção": "university_sa_eic",
    "Saúde e Proteção Social": "university_sa_hsp",
    "Serviços": "university_sa_s",
    "Year": "year",
    "Total": "total",


})
# Casting columns to correct type
df_unistudents_area["year"] = df_unistudents_area["year"].astype(int)
# Cast all columns after the 3rd one to numeric
df_unistudents_area.iloc[:, 3:] = df_unistudents_area.iloc[:, 3:].apply(pd.to_numeric, errors="coerce")
# Selecting decided type and timeframe
df_unistudents_area = df_unistudents_area[(df_unistudents_area["geographic_level"] == "Município") &(df_unistudents_area["year"] >= 2006)]
# Dropping uneeded column
df_unistudents_area = df_unistudents_area.drop(columns=["geographic_level","total"])


In [66]:
df_unistudents_area

,territory,year,university_sa_a,university_sa_ah,university_sa_sscl,university_sa_smi,university_sa_e,university_sa_eic,university_sa_hsp,university_sa_s
0,Abrantes,2014,0,74,82,68,0,104,0,0
1,Abrantes,2015,0,62,75,58,0,105,0,0
2,Abrantes,2016,0,56,85,59,0,113,0,0
3,Abrantes,2017,0,70,84,68,0,91,0,0
4,Abrantes,2018,0,78,86,44,0,115,0,0
...,...,...,...,...,...,...,...,...,...,...
3691,Óbidos,2021,0,0,0,0,0,0,0,0
3692,Óbidos,2022,0,0,0,0,0,0,0,0
3693,Óbidos,2023,0,0,0,0,0,0,0,0
3694,Óbidos,2024,0,0,0,0,0,0,0,0


# 4 Creating Datasets

## 4 | 1 Dataset | Population

In [67]:
df_population = (
    df_respop
    .merge(df_popden, on=['territory', 'year'], how='outer')
    .merge(df_ppower_pca, on=['territory', 'year'], how='outer')
)

In [68]:
df_population.columns

Index(['territory', 'year', 'population_total', 'population_density',
       'population_pp_pc'],
      dtype='object')

In [69]:
df_population


,territory,year,population_total,population_density,population_pp_pc
0,Abrantes,2006,40817,NaN,NaN
1,Abrantes,2007,40534,NaN,86.9
2,Abrantes,2008,40255,NaN,NaN
3,Abrantes,2009,39959,55.7,86.0
4,Abrantes,2010,39637,55.2,NaN
...,...,...,...,...,...
5847,Óbidos,2020,11875,85.0,NaN
5848,Óbidos,2021,12224,87.7,81.3
5849,Óbidos,2022,12619,90.6,NaN
5850,Óbidos,2023,13061,93.9,81.7


## 4 | 2 Dataset | Unemployment

In [70]:
df_unemployment = (
    df_unemp_g
    .merge(df_unemp_el, on=['territory', 'year'], how='outer')
    .merge(df_unemp_a, on=['territory', 'year'], how='outer')
)
df_unemployment = df_unemployment[[
    "territory",
    "year",
    "unemployment_total",
    "unemployment_female",
    "unemployment_male",
    "unemployment[<25]",
    "unemployment[25-34]",
    "unemployment[35-44]",
    "unemployment[45-54]",
    "unemployment[>54]",
    "unemployment_el_b1",
    "unemployment_el_b2",
    "unemployment_el_b3",
    "unemployment_el_hs",
    "unemployment_el_c",
    "unemployment_el_ne"
]]


In [71]:
df_unemployment

,territory,year,unemployment_total,unemployment_female,unemployment_male,unemployment[<25],unemployment[25-34],unemployment[35-44],unemployment[45-54],unemployment[>54],unemployment_el_b1,unemployment_el_b2,unemployment_el_b3,unemployment_el_hs,unemployment_el_c,unemployment_el_ne
0,Abrantes,2009,2164.1,1178.6,985.5,331.1,589.7,439.2,424.8,379.3,NaN,NaN,NaN,NaN,NaN,NaN
1,Abrantes,2010,2160.4,1107.9,1052.5,306.9,537.0,469.0,469.8,377.7,539.0,413.2,497.4,408.8,189.4,112.6
2,Abrantes,2011,2284.0,1170.0,1114.0,300.3,581.1,519.3,471.6,411.8,514.9,394.7,561.5,495.3,220.9,96.7
3,Abrantes,2012,2685.8,1380.4,1305.3,373.3,696.6,661.1,521.5,433.3,524.8,448.9,677.2,646.0,293.6,95.3
4,Abrantes,2013,2649.0,1324.8,1324.3,347.2,629.2,628.1,558.9,485.7,538.3,429.5,636.6,629.0,301.7,114.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5231,Óbidos,2021,272.2,168.3,103.9,26.5,44.2,52.3,67.1,82.1,33.8,32.3,57.7,101.4,43.3,3.7
5232,Óbidos,2022,217.7,134.2,83.5,24.6,29.6,37.7,50.8,75.0,30.8,28.3,46.3,71.8,36.6,3.8
5233,Óbidos,2023,208.4,126.1,82.3,23.1,26.6,42.8,43.8,72.2,27.5,25.1,43.0,74.3,31.9,6.7
5234,Óbidos,2024,202.0,117.7,84.3,21.6,31.5,39.1,41.6,68.3,22.1,24.4,35.0,81.5,32.8,6.2


## 4 | 3 Dataset | Company

In [72]:
df_company = (
    df_company_den
    .merge(df_company_lt, on=['territory', 'year'], how='outer')
    .merge(df_company_as, on=['territory', 'year'], how='outer')
    .merge(df_company_empnr, on=['territory', 'year'], how='outer')
    .merge(df_company_emp_as, on=['territory', 'year'], how='outer')
    .merge(df_company_gva_as, on=['territory', 'year'], how='outer')
    .merge(df_company_births_as, on=['territory', 'year'], how='outer')
    
)

In [73]:
df_company = df_company[[
    "territory", 
    "year", 
    "company_total",
    "company_individual",
    "company_society",
    "company_density",
    "company_nr_cae_a",
    "company_nr_cae_b",
    "company_nr_cae_c",
    "company_nr_cae_d",
    "company_nr_cae_e",
    "company_nr_cae_f",
    "company_nr_cae_g",
    "company_nr_cae_h",
    "company_nr_cae_i",
    "company_nr_cae_j",
    "company_nr_cae_l",
    "company_nr_cae_m",
    "company_nr_cae_n",
    "company_nr_cae_p",
    "company_nr_cae_q",
    "company_nr_cae_r",
    "company_nr_cae_s",
    "company_nr_employee_[<10]",
    "company_nr_employee_[10-19]",
    "company_nr_employee_[20-49]",
    "company_nr_employee_[50-249]",
    "company_nr_employee_[>249]",
    "company_employee_nr_cae_a",
    "company_employee_nr_cae_b",
    "company_employee_nr_cae_c",
    "company_employee_nr_cae_d",
    "company_employee_nr_cae_e",
    "company_employee_nr_cae_f",
    "company_employee_nr_cae_g",
    "company_employee_nr_cae_h",
    "company_employee_nr_cae_i",
    "company_employee_nr_cae_j",
    "company_employee_nr_cae_l",
    "company_employee_nr_cae_m",
    "company_employee_nr_cae_n",
    "company_employee_nr_cae_p",
    "company_employee_nr_cae_q",
    "company_employee_nr_cae_r",
    "company_employee_nr_cae_s",
    "company_employee_nr_total",
    "company_gva_cae_a",
    "company_gva_cae_b",
    "company_gva_cae_c",
    "company_gva_cae_d",
    "company_gva_cae_e",
    "company_gva_cae_f",
    "company_gva_cae_g",
    "company_gva_cae_h",
    "company_gva_cae_i",
    "company_gva_cae_j",
    "company_gva_cae_l",
    "company_gva_cae_m",
    "company_gva_cae_n",
    "company_gva_cae_p",
    "company_gva_cae_q",
    "company_gva_cae_r",
    "company_gva_cae_s",
    "company_gva_total",
    "company_births_af",
    "company_births_ice",
    "company_births_s",
    "company_births_total",
]]


In [74]:
df_company

,territory,year,company_total,company_individual,company_society,company_density,company_nr_cae_a,company_nr_cae_b,company_nr_cae_c,company_nr_cae_d,...,company_gva_cae_n,company_gva_cae_p,company_gva_cae_q,company_gva_cae_r,company_gva_cae_s,company_gva_total,company_births_af,company_births_ice,company_births_s,company_births_total
0,Abrantes,2009,3298,2327,971,4.6,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,14,29,369,412
1,Abrantes,2010,3079,2147,932,4.3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,27,25,282,334
2,Abrantes,2011,3081,2143,938,4.3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,25,32,299,356
3,Abrantes,2012,2922,2002,920,4.1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,23,33,287,343
4,Abrantes,2013,2838,1938,900,4.0,252,0,150,5,...,4078,954,5229,351,1286,220136,71,31,276,378
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4923,Óbidos,2020,1862,1281,581,13.2,354,1,89,17,...,2537,743,1483,254,786,77110,20,21,134,175
4924,Óbidos,2021,1984,1386,598,14.0,350,1,90,14,...,2798,601,1864,690,667,77426,19,22,150,191
4925,Óbidos,2022,2257,1623,634,15.9,338,1,91,11,...,3804,901,2343,1586,674,112613,11,31,304,346
4926,Óbidos,2023,2343,1682,661,16.6,326,1,89,5,...,4471,1122,2857,3422,824,125715,16,28,321,365


## 4 | 4 Dataset | University

In [75]:
df_university = (
    df_unistudents_type
    .merge(df_unipublicstudents_type, on=['territory', 'year'], how='outer')
    .merge(df_uniprivstudents_type, on=['territory', 'year'], how='outer')
    .merge(df_unistudents_gradlevel, on=['territory', 'year'], how='outer')
    .merge(df_unistudents_area, on=['territory', 'year'], how='outer')
)

In [76]:
df_university = df_university[[
    "territory", 
    "year", 
    "university_total",
    "university_poli",
    "university_uni",
    "university_public_total",
    "university_public_poli",
    "university_public_uni",
    "university_private_total",
    "university_private_poli",
    "university_private_uni",
    "university_gl_lic",
    "university_gl_lic1",
    "university_gl_ma",
    "university_gl_phd",
    "university_gl_spc",
    "university_gl_esp",
    "university_gl_fc",
    "university_gl_ba",
    "university_gl_htc",
    "university_sa_a",
    "university_sa_ah",
    "university_sa_sscl",
    "university_sa_smi",
    "university_sa_e",
    "university_sa_eic",
    "university_sa_hsp",
    "university_sa_s",
]]


In [77]:
df_university.columns

Index(['territory', 'year', 'university_total', 'university_poli',
       'university_uni', 'university_public_total', 'university_public_poli',
       'university_public_uni', 'university_private_total',
       'university_private_poli', 'university_private_uni',
       'university_gl_lic', 'university_gl_lic1', 'university_gl_ma',
       'university_gl_phd', 'university_gl_spc', 'university_gl_esp',
       'university_gl_fc', 'university_gl_ba', 'university_gl_htc',
       'university_sa_a', 'university_sa_ah', 'university_sa_sscl',
       'university_sa_smi', 'university_sa_e', 'university_sa_eic',
       'university_sa_hsp', 'university_sa_s'],
      dtype='object')

In [78]:
df_university

,territory,year,university_total,university_poli,university_uni,university_public_total,university_public_poli,university_public_uni,university_private_total,university_private_poli,...,university_gl_ba,university_gl_htc,university_sa_a,university_sa_ah,university_sa_sscl,university_sa_smi,university_sa_e,university_sa_eic,university_sa_hsp,university_sa_s
0,Abrantes,2009,455,455,0,455,455,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Abrantes,2010,458,458,0,458,458,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Abrantes,2011,429,429,0,429,429,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Abrantes,2012,402,402,0,402,402,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Abrantes,2013,372,372,0,372,372,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5231,Óbidos,2021,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5232,Óbidos,2022,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5233,Óbidos,2023,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5234,Óbidos,2024,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# 5 Dataset Validation

## 5 | 1 Validation | Functions created

In [79]:
# Create a function to check for which year we have data for any given metric
def df_infometrics(df, expected_territories=308):
    """
    Creates a completeness matrix:
    1  - all territories present
    0  - no territories present
    -1 - partial coverage
    rows = years
    columns = metric columns
    """

    # metric columns
    metric_cols = [col for col in df.columns
                   if col not in ['territory', 'year']]

    result = {}

    for col in metric_cols:
        counts = (
            df[df[col].notna()]
            .groupby('year')['territory']
            .nunique()
        )

        # transform values
        counts = counts.apply(
            lambda x: 1 if x == expected_territories
            else 0 if x == 0
            else -1
        )

        result[col] = counts

    # create dataframe
    result_df = pd.DataFrame(result).fillna(0).astype(int)

    return result_df

In [80]:
# Create a function to color code the result
def color_map(val):
    """
    Color rules:
    1  - green | all territories present
    0  - red | no territories present
    -1 - yellow | partial coverage
    """

    if val == 1:
        return 'background-color: green'
    elif val == 0:
        return 'background-color: red'
    elif val == -1:
        return 'background-color: yellow'
    
    return ''


## 5 | 2 Validation

In [81]:
df_population_info = df_infometrics(df_population)
#df_population_info
df_population_info.style.map(color_map)

,population_total,population_density,population_pp_pc
year,,,
2006,1,0,0
2007,1,0,1
2008,1,0,0
2009,1,1,1
2010,1,1,0
2011,1,1,1
2012,1,1,0
2013,1,1,1
2014,1,1,0


In [82]:
df_unemployment_info = df_infometrics(df_unemployment)
#df_unemployment_info
df_unemployment_info.style.map(color_map)

,unemployment_total,unemployment_female,unemployment_male,unemployment[<25],unemployment[25-34],unemployment[35-44],unemployment[45-54],unemployment[>54],unemployment_el_b1,unemployment_el_b2,unemployment_el_b3,unemployment_el_hs,unemployment_el_c,unemployment_el_ne
year,,,,,,,,,,,,,,
2009,1,1,1,1,1,1,1,1,0,0,0,0,0,0
2010,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2011,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2012,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2013,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2014,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2015,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2016,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2017,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [83]:
df_company_info = df_infometrics(df_company)
#df_company_info
df_company_info.style.map(color_map)

,company_total,company_individual,company_society,company_density,company_nr_cae_a,company_nr_cae_b,company_nr_cae_c,company_nr_cae_d,company_nr_cae_e,company_nr_cae_f,company_nr_cae_g,company_nr_cae_h,company_nr_cae_i,company_nr_cae_j,company_nr_cae_l,company_nr_cae_m,company_nr_cae_n,company_nr_cae_p,company_nr_cae_q,company_nr_cae_r,company_nr_cae_s,company_nr_employee_[<10],company_nr_employee_[10-19],company_nr_employee_[20-49],company_nr_employee_[50-249],company_nr_employee_[>249],company_employee_nr_cae_a,company_employee_nr_cae_b,company_employee_nr_cae_c,company_employee_nr_cae_d,company_employee_nr_cae_e,company_employee_nr_cae_f,company_employee_nr_cae_g,company_employee_nr_cae_h,company_employee_nr_cae_i,company_employee_nr_cae_j,company_employee_nr_cae_l,company_employee_nr_cae_m,company_employee_nr_cae_n,company_employee_nr_cae_p,company_employee_nr_cae_q,company_employee_nr_cae_r,company_employee_nr_cae_s,company_employee_nr_total,company_gva_cae_a,company_gva_cae_b,company_gva_cae_c,company_gva_cae_d,company_gva_cae_e,company_gva_cae_f,company_gva_cae_g,company_gva_cae_h,company_gva_cae_i,company_gva_cae_j,company_gva_cae_l,company_gva_cae_m,company_gva_cae_n,company_gva_cae_p,company_gva_cae_q,company_gva_cae_r,company_gva_cae_s,company_gva_total,company_births_af,company_births_ice,company_births_s,company_births_total
year,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2009,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1
2010,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1
2011,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1
2012,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1
2013,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2014,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2015,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2016,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2017,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [84]:
df_university_info = df_infometrics(df_university)
#df_university_info
df_university_info.style.map(color_map)

,university_total,university_poli,university_uni,university_public_total,university_public_poli,university_public_uni,university_private_total,university_private_poli,university_private_uni,university_gl_lic,university_gl_lic1,university_gl_ma,university_gl_phd,university_gl_spc,university_gl_esp,university_gl_fc,university_gl_ba,university_gl_htc,university_sa_a,university_sa_ah,university_sa_sscl,university_sa_smi,university_sa_e,university_sa_eic,university_sa_hsp,university_sa_s
year,,,,,,,,,,,,,,,,,,,,,,,,,,
2009,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2010,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2011,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2012,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2013,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2014,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1
2015,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1
2016,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2017,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


## 5 | 3 Validation | Conclusion

1-Datasets will be selected between years 2016-2024 <br>
2-All metrics are absolute values, so to be able to compare across territories, paralel datasets will be created with metrics per 10k residents <br>
3-Population Dataset <br>
     -No need to normalize, [population_total] is comparable per se, and [population_density] is per km2, and [population_pp_pc] is compared with global average <br>

# 6 Dataset Normalization

## 6 | 1 Normalization | Functions created

In [85]:
def convert_to_p10k(df, exclude_cols=None):
    """
    Convert all metric columns in a dataframe to per 10k population.
    """

    if exclude_cols is None:
        exclude_cols = []

    # make a copy so original dataframe is not modified
    df_result = df.copy()

    # base columns that should never be transformed
    fixed_cols = ['territory', 'year', 'population_total'] + exclude_cols

    # columns to transform
    metric_cols = [
        col for col in df_result.columns
        if col not in fixed_cols
    ]

    # convert metrics to per 10k population
    df_result[metric_cols] = (
        df_result[metric_cols]
        .div(df_result['population_total'] / 10000, axis=0)
        .round(2)
    )

    return df_result

## 6 | 2 Normalization | Year

In [86]:
df_population = df_population[(df_population['year'] >= 2016) & (df_population['year'] <= 2024)]
df_unemployment = df_unemployment[(df_unemployment['year'] >= 2016) & (df_unemployment['year'] <= 2024)]
df_company = df_company[(df_company['year'] >= 2016) & (df_company['year'] <= 2024)]
df_university = df_university[(df_university['year'] >= 2016) & (df_university['year'] <= 2024)]

## 6 | 3 Normalization | p10k Datasets

### 6 | 3 | 1 Normalization | Unemployment

In [87]:
# join the datasets on territory and year to get the territory resident population
df_unemployment_p10k = df_population.merge(
    df_unemployment,
    on=['territory', 'year'],
    how='inner'   # use 'left' if you want to keep all rows from df_population
)
#drop uneeded columns
df_unemployment_p10k = df_unemployment_p10k.drop(columns=['population_density', 'population_pp_pc'])

#convert metrics to per 10k residents
df_unemployment_p10k = convert_to_p10k(df_unemployment_p10k)

df_unemployment_p10k

,territory,year,population_total,unemployment_total,unemployment_female,unemployment_male,unemployment[<25],unemployment[25-34],unemployment[35-44],unemployment[45-54],unemployment[>54],unemployment_el_b1,unemployment_el_b2,unemployment_el_b3,unemployment_el_hs,unemployment_el_c,unemployment_el_ne
0,Abrantes,2016,36421,512.643804,270.970045,241.67376,58.83968,101.672112,123.774745,119.052195,109.305071,90.826721,78.114275,123.994399,131.160594,58.620027,29.955246
1,Abrantes,2017,35937,455.769819,243.175557,212.594262,50.171133,90.269082,105.406684,107.994546,101.928375,70.707071,69.510532,115.591173,115.53552,54.345104,30.080419
2,Abrantes,2018,35463,397.34371,219.637369,177.706342,49.037025,74.415588,87.386854,93.618701,92.857344,61.613513,60.42918,95.282407,106.110594,45.850605,28.08561
3,Abrantes,2019,35083,387.196078,216.572129,170.652453,47.401876,72.485249,83.772767,85.625517,97.939173,53.957757,61.226235,101.644671,105.5782,42.299689,22.518029
4,Abrantes,2020,34745,435.458339,228.752338,206.706001,51.892359,89.538063,93.04936,91.178587,109.828752,57.33199,63.807742,111.382933,132.076558,47.517628,23.370269
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2767,Óbidos,2020,11875,233.094737,145.263158,87.831579,24.926316,44.042105,46.231579,52.8,65.094737,30.4,32.421053,48.926316,78.989474,36.8,5.642105
2768,Óbidos,2021,12224,222.676702,137.679974,84.996728,21.678665,36.158377,42.784686,54.892016,67.162958,27.650524,26.423429,47.202225,82.951571,35.42212,3.026832
2769,Óbidos,2022,12619,172.517632,106.347571,66.170061,19.494413,23.456692,29.875584,40.256756,59.434187,24.407639,22.4265,36.690704,56.898328,29.003883,3.011332
2770,Óbidos,2023,13061,159.558992,96.546972,63.012021,17.686241,20.365975,32.769313,33.534951,55.279075,21.055049,19.217518,32.922441,56.886915,24.423857,5.129776


### 6 | 3 | 2 Normalization | Company

In [88]:
# join the datasets on territory and year to get the territory resident population
df_company_p10k = df_population.merge(
    df_company,
    on=['territory', 'year'],
    how='inner'   # use 'left' if you want to keep all rows from df_population
)
#drop uneeded columns
df_company_p10k = df_company_p10k.drop(columns=['population_density', 'population_pp_pc'])

#convert metrics to per 10k residents
df_company_p10k = convert_to_p10k(df_company_p10k,exclude_cols=['company_density'])

df_company_p10k

,territory,year,population_total,company_total,company_individual,company_society,company_density,company_nr_cae_a,company_nr_cae_b,company_nr_cae_c,...,company_gva_cae_n,company_gva_cae_p,company_gva_cae_q,company_gva_cae_r,company_gva_cae_s,company_gva_total,company_births_af,company_births_ice,company_births_s,company_births_total
0,Abrantes,2016,36421,834.957854,587.02397,247.933884,4.3,83.742896,0.0,40.086763,...,943.686335,292.688284,1694.077593,117.789188,442.052662,71508.195821,15.101178,20.592515,96.922105,132.615799
1,Abrantes,2017,35937,873.751287,620.808637,252.94265,4.4,84.870746,0.0,43.965829,...,958.900298,294.125831,1781.450872,105.740602,461.641205,74182.040794,13.634972,13.634972,101.288366,128.55831
2,Abrantes,2018,35463,867.66489,611.341398,256.323492,4.3,87.133068,0.0,43.98951,...,803.654513,311.028396,1783.831035,124.354962,409.72281,75449.341567,13.817218,10.433409,78.391563,102.64219
3,Abrantes,2019,35083,879.058233,609.126928,269.931306,4.3,86.651655,0.0,45.321096,...,764.472822,304.705983,1977.310948,144.229399,449.790497,75121.853889,14.821994,8.55115,90.642191,114.015335
4,Abrantes,2020,34745,889.624406,608.145057,281.47935,4.3,86.055548,0.0,42.308246,...,643.545834,262.195999,2000.0,90.660527,539.358181,69422.36293,9.785581,7.19528,90.372715,107.353576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2767,Óbidos,2020,11875,1568.0,1078.736842,489.263158,13.2,298.105263,0.842105,74.947368,...,2136.421053,625.684211,1248.842105,213.894737,661.894737,64934.736842,16.842105,17.684211,112.842105,147.368421
2768,Óbidos,2021,12224,1623.036649,1133.835079,489.201571,14.0,286.32199,0.818063,73.625654,...,2288.939791,491.655759,1524.86911,564.463351,545.647906,63339.332461,15.543194,17.997382,122.709424,156.25
2769,Óbidos,2022,12619,1788.572787,1286.155797,502.41699,15.9,267.850067,0.792456,72.11348,...,3014.501942,714.002694,1856.723988,1256.834931,534.115223,89240.827324,8.717014,24.56613,240.906569,274.189714
2770,Óbidos,2023,13061,1793.890207,1287.803384,506.086823,16.6,249.59804,0.765638,68.141796,...,3423.168211,859.046015,2187.428221,2620.013781,630.885843,96252.20121,12.250211,21.437868,245.769849,279.457928


### 6 | 3 | 3 Normalization | University

In [89]:
# join the datasets on territory and year to get the territory resident population
df_university_p10k = df_population.merge(
    df_university,
    on=['territory', 'year'],
    how='inner'   # use 'left' if you want to keep all rows from df_population
)
#drop uneeded columns
df_university_p10k = df_university_p10k.drop(columns=['population_density', 'population_pp_pc'])

#convert metrics to per 10k residents
df_university_p10k = convert_to_p10k(df_university_p10k)

df_university_p10k

,territory,year,population_total,university_total,university_poli,university_uni,university_public_total,university_public_poli,university_public_uni,university_private_total,...,university_gl_ba,university_gl_htc,university_sa_a,university_sa_ah,university_sa_sscl,university_sa_smi,university_sa_e,university_sa_eic,university_sa_hsp,university_sa_s
0,Abrantes,2016,36421,85.939431,85.939431,0.0,85.939431,85.939431,0.0,0.0,...,0.0,9.335274,0.0,15.375745,23.338184,16.199445,0.0,31.026056,0.0,0.0
1,Abrantes,2017,35937,87.096864,87.096864,0.0,87.096864,87.096864,0.0,0.0,...,0.0,18.643738,0.0,19.478532,23.374238,18.922002,0.0,25.322091,0.0,0.0
2,Abrantes,2018,35463,91.080845,91.080845,0.0,91.080845,91.080845,0.0,0.0,...,0.0,21.430787,0.0,21.994755,24.250627,12.407298,0.0,32.428165,0.0,0.0
3,Abrantes,2019,35083,102.328763,102.328763,0.0,102.328763,102.328763,0.0,0.0,...,0.0,31.639255,0.0,25.938489,25.083374,20.52276,0.0,30.78414,0.0,0.0
4,Abrantes,2020,34745,125.19787,125.19787,0.0,125.19787,125.19787,0.0,0.0,...,0.0,47.201036,0.0,37.415455,31.083609,28.493308,0.0,28.205497,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2767,Óbidos,2020,11875,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2768,Óbidos,2021,12224,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2769,Óbidos,2022,12619,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2770,Óbidos,2023,13061,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 7 Datasets Export

In [90]:

df_nuts.to_csv("../data/processed_data/00_df_nuts.csv", index=False)
df_population.to_csv("../data/processed_data/01_df_population.csv", index=False)
df_unemployment.to_csv("../data/processed_data/02_df_unemployment.csv", index=False)
df_unemployment_p10k.to_csv("../data/processed_data/02_df_unemployment_p10k.csv", index=False)
df_company.to_csv("../data/processed_data/03_df_company.csv", index=False)
df_company_p10k.to_csv("../data/processed_data/03_df_company_p10k.csv", index=False)
df_university.to_csv("../data/processed_data/04_df_university.csv", index=False)
df_university_p10k.to_csv("../data/processed_data/04_df_university_p10k.csv", index=False)

